In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [106]:
transaction_df = pd.read_parquet("transactions.parquet")
transaction_df = transaction_df.dropna()
transaction_df['category'] = transaction_df['category'].astype(int)

consumer_df = pd.read_parquet("consumer_data.parquet")
categories_df = pd.read_csv("transaction_categories.csv")


In [41]:
consumer_df

,evaluation_date,FPF_TARGET,total_balance,masked_consumer_id
0,2021-08-26,0.0,1380.84,C01100001
1,2022-08-02,0.0,20163.90,C01100002
2,2021-03-04,0.0,3986.25,C01100003
3,2022-11-19,0.0,5956.03,C01100004
4,2021-11-21,0.0,29421.10,C01100005
...,...,...,...,...
4995,2023-03-31,0.0,1412.80,C04104996
4996,2023-03-31,0.0,-439.21,C04104997
4997,2023-03-31,0.0,84.53,C04104998
4998,2023-03-31,0.0,52655.88,C04104999


In [4]:
cat_map = categories_df.set_index('code')['description'].to_dict()


In [5]:
transaction_df['category_name'] = transaction_df['category'].apply(lambda x: cat_map[x])

In [6]:
consumer_df['evaluation_date'] = pd.to_datetime(consumer_df['evaluation_date'], unit = 'ns')
transaction_df['posted_date'] = pd.to_datetime(transaction_df['posted_date'], unit = 'ns')


In [7]:
consumer_df[consumer_df['FPF_TARGET']==1]

,evaluation_date,FPF_TARGET,total_balance,masked_consumer_id
174,2022-03-31,1.0,39534.66,C01100175
175,2021-08-03,1.0,33788.73,C01100176
417,2022-04-23,1.0,9472.89,C01100418
479,2021-11-29,1.0,1704.32,C01100480
783,2022-05-03,1.0,9773.72,C01100784
...,...,...,...,...
4840,2023-03-21,1.0,4192.67,C04104841
4882,2023-03-24,1.0,148.69,C04104883
4888,2023-03-24,1.0,1070.53,C04104889
4918,2023-03-27,1.0,6742.72,C04104919


In [8]:
consumer_df[consumer_df['FPF_TARGET']==0]

,evaluation_date,FPF_TARGET,total_balance,masked_consumer_id
0,2021-08-26,0.0,1380.84,C01100001
1,2022-08-02,0.0,20163.90,C01100002
2,2021-03-04,0.0,3986.25,C01100003
3,2022-11-19,0.0,5956.03,C01100004
4,2021-11-21,0.0,29421.10,C01100005
...,...,...,...,...
4995,2023-03-31,0.0,1412.80,C04104996
4996,2023-03-31,0.0,-439.21,C04104997
4997,2023-03-31,0.0,84.53,C04104998
4998,2023-03-31,0.0,52655.88,C04104999


## Looking at categories

I want to understand these categories.
Some interesting categories, which should correlate with "being bad with money"

Bad: UNEMPLOYMENT_BENEFITS, SMALL_DOLLAR_ADVANCE, OVERDRAFT, BNPL

Good: Investment income, Paycheck/paycheck_placeholder

In [9]:
categories = transaction_df['category'].unique()

for i, category in enumerate(categories):
    category_data = transaction_df[transaction_df['category'] == category]['amount']
    print(f'Category: {cat_map[category]}')
    print(category_data.describe())
    print('_________________________')


Category: HEALTHCARE_MEDICAL
count    238626.000000
mean        -51.696584
std         297.854137
min      -82500.000000
25%         -45.130000
50%         -21.360000
75%          -9.440000
max          -0.000000
Name: amount, dtype: float64
_________________________
Category: ESSENTIAL_SERVICES
count    424454.000000
mean       -145.646223
std         374.107225
min      -49234.290000
25%        -174.767500
50%         -90.150000
75%         -39.530000
max          -0.000000
Name: amount, dtype: float64
_________________________
Category: GENERAL_MERCHANDISE
count    3.069621e+06
mean    -5.481288e+01
std      3.283958e+02
min     -1.830000e+05
25%     -4.960000e+01
50%     -2.032000e+01
75%     -9.990000e+00
max     -0.000000e+00
Name: amount, dtype: float64
_________________________
Category: SELF_TRANSFER
count    1.015913e+06
mean     1.657626e+02
std      4.304083e+03
min     -1.051734e+06
25%     -4.400000e-01
50%      5.000000e+00
75%      8.500000e+01
max      1.051734e+06
Nam

In [ ]:
categories = transaction_df['category'].unique()
num_categories = len(categories)
fig, axes = plt.subplots(nrows=num_categories, ncols=1, figsize=(8, 4 * num_categories))
for i, category in enumerate(categories):
    category_data = transaction_df[transaction_df['category'] == category]['amount']
    axes[i].hist(category_data, bins=10, edgecolor='black')  # You can adjust the number of bins
    axes[i].set_title(f'Category: {cat_map[category]}')
    axes[i].set_xlabel('Amount')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


## Looking at people

### Default

In [11]:
transaction_df[transaction_df['masked_consumer_id']=='C01100480'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
545634,C01100480,2021-05-21,-125000.00,1.0,C01T0545634,EXTERNAL_TRANSFER
545600,C01100480,2021-10-20,-40000.00,1.0,C01T0545600,EXTERNAL_TRANSFER
545526,C01100480,2021-10-14,-40000.00,19.0,C01T0545526,ATM_CASH
545346,C01100480,2021-04-05,-40000.00,1.0,C01T0545346,EXTERNAL_TRANSFER
546108,C01100480,2021-08-26,-40000.00,1.0,C01T0546108,EXTERNAL_TRANSFER
...,...,...,...,...,...,...
545234,C01100480,2021-07-15,69500.00,5.0,C01T0545234,PAYCHECK_PLACEHOLDER
545132,C01100480,2021-04-01,71434.95,5.0,C01T0545132,PAYCHECK_PLACEHOLDER
545081,C01100480,2021-08-26,99041.26,7.0,C01T0545081,INVESTMENT_INCOME
545186,C01100480,2021-10-08,120534.53,2.0,C01T0545186,DEPOSIT


In [12]:
transaction_df[transaction_df['masked_consumer_id']=='C01102732'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
3082603,C01102732,2021-02-22,-12000.00,19.0,C01T03082603,ATM_CASH
3082970,C01102732,2020-10-20,-11800.00,19.0,C01T03082970,ATM_CASH
3082647,C01102732,2020-11-20,-11000.00,19.0,C01T03082647,ATM_CASH
3082492,C01102732,2020-09-18,-9960.00,19.0,C01T03082492,ATM_CASH
3082443,C01102732,2020-06-19,-9600.00,19.0,C01T03082443,ATM_CASH
...,...,...,...,...,...,...
3082226,C01102732,2020-08-20,13364.95,5.0,C01T03082226,PAYCHECK_PLACEHOLDER
3082328,C01102732,2020-09-18,13417.85,5.0,C01T03082328,PAYCHECK_PLACEHOLDER
3082224,C01102732,2020-11-20,14021.29,5.0,C01T03082224,PAYCHECK_PLACEHOLDER
3082290,C01102732,2020-06-22,24000.00,2.0,C01T03082290,DEPOSIT


In [13]:
transaction_df[transaction_df['masked_consumer_id']=='C01102806'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
3171016,C01102806,2020-12-16,-15700.00,1.0,C01T03171016,EXTERNAL_TRANSFER
3171061,C01102806,2021-03-15,-6000.00,12.0,C01T03171061,LOAN
3170956,C01102806,2020-11-30,-5000.00,1.0,C01T03170956,EXTERNAL_TRANSFER
3170995,C01102806,2020-12-01,-3029.03,1.0,C01T03170995,EXTERNAL_TRANSFER
3171011,C01102806,2021-10-28,-3000.00,1.0,C01T03171011,EXTERNAL_TRANSFER
...,...,...,...,...,...,...
3170901,C01102806,2021-07-28,5883.28,5.0,C01T03170901,PAYCHECK_PLACEHOLDER
3170856,C01102806,2021-09-28,6129.02,5.0,C01T03170856,PAYCHECK_PLACEHOLDER
3170860,C01102806,2020-12-29,6594.81,5.0,C01T03170860,PAYCHECK_PLACEHOLDER
3170826,C01102806,2020-11-25,7707.14,5.0,C01T03170826,PAYCHECK_PLACEHOLDER


In [14]:
transaction_df[transaction_df['masked_consumer_id']=='C01102654'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
2995835,C01102654,2021-11-10,-14400.0,1.0,C01T02995835,EXTERNAL_TRANSFER
2995806,C01102654,2021-12-03,-8700.0,1.0,C01T02995806,EXTERNAL_TRANSFER
2995774,C01102654,2021-11-09,-4900.0,1.0,C01T02995774,EXTERNAL_TRANSFER
2995856,C01102654,2021-11-04,-4600.0,1.0,C01T02995856,EXTERNAL_TRANSFER
2995788,C01102654,2021-11-16,-3700.0,1.0,C01T02995788,EXTERNAL_TRANSFER
...,...,...,...,...,...,...
2995743,C01102654,2021-11-16,3798.0,4.0,C01T02995743,MISCELLANEOUS
2995741,C01102654,2021-11-02,5000.0,4.0,C01T02995741,MISCELLANEOUS
2995738,C01102654,2021-11-09,5000.0,4.0,C01T02995738,MISCELLANEOUS
2995750,C01102654,2021-12-03,9500.0,1.0,C01T02995750,EXTERNAL_TRANSFER


### Not default

In [15]:

transaction_df[transaction_df['masked_consumer_id']=='C01100006'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
9678,C01100006,2022-11-14,-5216.80,1.0,C01T09678,EXTERNAL_TRANSFER
10601,C01100006,2022-03-28,-4914.00,1.0,C01T010601,EXTERNAL_TRANSFER
10652,C01100006,2022-04-21,-4500.00,1.0,C01T010652,EXTERNAL_TRANSFER
10539,C01100006,2022-06-02,-3600.00,1.0,C01T010539,EXTERNAL_TRANSFER
10911,C01100006,2022-04-18,-3501.00,1.0,C01T010911,EXTERNAL_TRANSFER
...,...,...,...,...,...,...
9316,C01100006,2022-06-02,3866.61,3.0,C01T09316,PAYCHECK
9335,C01100006,2022-04-21,4500.00,0.0,C01T09335,SELF_TRANSFER
9422,C01100006,2022-04-21,4728.10,3.0,C01T09422,PAYCHECK
9543,C01100006,2022-03-28,4914.00,0.0,C01T09543,SELF_TRANSFER


In [16]:
transaction_df[transaction_df['masked_consumer_id']=='C01100036'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
39032,C01100036,2021-11-22,-20000.00,12.0,C01T039032,LOAN
39073,C01100036,2022-02-28,-10000.00,1.0,C01T039073,EXTERNAL_TRANSFER
39010,C01100036,2022-02-28,-8630.31,26.0,C01T039010,CREDIT_CARD_PAYMENT
39196,C01100036,2021-11-19,-7000.00,19.0,C01T039196,ATM_CASH
39098,C01100036,2021-08-30,-6331.51,26.0,C01T039098,CREDIT_CARD_PAYMENT
...,...,...,...,...,...,...
38941,C01100036,2021-05-21,19456.53,3.0,C01T038941,PAYCHECK
38995,C01100036,2021-09-23,21704.94,7.0,C01T038995,INVESTMENT_INCOME
38946,C01100036,2021-08-27,42255.44,3.0,C01T038946,PAYCHECK
38942,C01100036,2022-02-25,46460.84,3.0,C01T038942,PAYCHECK


In [17]:
transaction_df[transaction_df['masked_consumer_id']=='C01100046'].sort_values(by='amount')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
46492,C01100046,2021-02-26,-9000.00,0.0,C01T046492,SELF_TRANSFER
47236,C01100046,2021-07-02,-2550.75,12.0,C01T047236,LOAN
46870,C01100046,2021-05-10,-2500.75,12.0,C01T046870,LOAN
46509,C01100046,2021-06-03,-2500.75,12.0,C01T046509,LOAN
46188,C01100046,2020-12-18,-2500.00,0.0,C01T046188,SELF_TRANSFER
...,...,...,...,...,...,...
45938,C01100046,2021-05-14,4227.19,5.0,C01T045938,PAYCHECK_PLACEHOLDER
45978,C01100046,2020-11-13,4227.20,5.0,C01T045978,PAYCHECK_PLACEHOLDER
45941,C01100046,2020-12-18,5219.89,5.0,C01T045941,PAYCHECK_PLACEHOLDER
46020,C01100046,2021-02-26,9000.00,0.0,C01T046020,SELF_TRANSFER


In [18]:

transaction_df[transaction_df['masked_consumer_id']=='C01100056'].sort_values(by='posted_date')

,masked_consumer_id,posted_date,amount,category,masked_transaction_id,category_name
56171,C01100056,2021-04-12,-26.00,12.0,C01T056171,LOAN
55554,C01100056,2021-04-12,-17.98,1.0,C01T055554,EXTERNAL_TRANSFER
55958,C01100056,2021-04-12,-0.56,23.0,C01T055958,ACCOUNT_FEES
56072,C01100056,2021-04-12,-0.98,23.0,C01T056072,ACCOUNT_FEES
56194,C01100056,2021-04-12,-21.00,12.0,C01T056194,LOAN
...,...,...,...,...,...,...
55810,C01100056,2022-04-04,-2500.00,26.0,C01T055810,CREDIT_CARD_PAYMENT
55824,C01100056,2022-04-06,-171.62,22.0,C01T055824,ESSENTIAL_SERVICES
54860,C01100056,2022-04-08,20.00,1.0,C01T054860,EXTERNAL_TRANSFER
55851,C01100056,2022-04-08,-20.00,1.0,C01T055851,EXTERNAL_TRANSFER


## Feature engineering

From the observations above, we're going to generate some features and see how they relate with FPF_target

In [107]:
transaction_df.head()

,masked_consumer_id,posted_date,amount,category,masked_transaction_id
2715977,C02103629,1660262400000000000,-21.00,27,C02T02715977
2715978,C02103629,1660003200000000000,-8.60,22,C02T02715978
2715979,C02103629,1659830400000000000,-30.05,16,C02T02715979
2715980,C02103629,1664409600000000000,-0.46,0,C02T02715980
2715981,C02103629,1658361600000000000,-50.00,12,C02T02715981


In [146]:
def count_positive(series):
    return (series > 0).sum()

def count_negative(series):
    return (series < 0).sum()

cat_gb = transaction_df.groupby(by=['masked_consumer_id', 'category'])['amount'].agg(mean='mean', median='median',max='max', min='min', positive_count = count_positive, negative_count = count_negative)


In [147]:
cat_gb = cat_gb.reset_index()

In [154]:
features = ['mean','median','positive_count','negative_count', 'max','min']

catf_df = cat_gb.melt(
    id_vars = ['masked_consumer_id', 'category'],
    value_vars = features
).pivot_table(
    index=['masked_consumer_id'],
    columns= ['category', 'variable']
)


catf_df=catf_df.reset_index()
catf_df.columns = ['_'.join(str(level) for level in col) for col in catf_df.columns]
catf_df = catf_df.rename(columns={'masked_consumer_id__': 'masked_consumer_id'})


In [155]:
feat_df = pd.merge(catf_df, consumer_df[['masked_consumer_id','FPF_TARGET']], how='left',on='masked_consumer_id')

In [156]:
corr = feat_df[feat_df.columns[1:-1]].corrwith(feat_df['FPF_TARGET'])

/Users/erlangsurya/miniforge3/envs/ML/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/erlangsurya/miniforge3/envs/ML/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/erlangsurya/miniforge3/envs/ML/lib/python3.12/site-packages/numpy/lib/function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/erlangsurya/miniforge3/envs/ML/lib/python3.12/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/erlangsurya/miniforge3/envs/ML/lib/python3.12/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


In [157]:
corr = corr.sort_values(ascending=False)

In [158]:
corr_chosen = corr[abs(corr)>0.1].index

In [159]:
corr_chosen

Index(['value_31_min', 'value_31_median', 'value_31_mean', 'value_31_max',
       'value_20_negative_count', 'value_5_median', 'value_5_mean',
       'value_15_negative_count', 'value_10_min', 'value_5_positive_count',
       'value_8_positive_count', 'value_1_negative_count',
       'value_7_positive_count', 'value_34_negative_count', 'value_10_max',
       'value_3_mean', 'value_10_median', 'value_16_negative_count',
       'value_10_mean', 'value_3_median', 'value_27_negative_count',
       'value_26_negative_count', 'value_14_negative_count',
       'value_18_negative_count', 'value_13_negative_count',
       'value_22_negative_count', 'value_31_negative_count',
       'value_3_positive_count'],
      dtype='object')

In [160]:
corr

value_31_min               0.121238
value_31_median            0.115610
value_31_mean              0.115329
value_31_max               0.110756
value_26_min               0.095089
                             ...   
value_36_mean                   NaN
value_36_median                 NaN
value_36_min                    NaN
value_36_negative_count         NaN
value_36_positive_count         NaN
Length: 222, dtype: float64